# Music Annotation Project V7.7F: Deterministic Structural-Policy Freeze

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook converts the retrospective V7.7 structural-risk analysis into a deterministic, serialised candidate policy that can score completely unseen V7.8 components.

## Final-fit and calibration procedure

The procedure deliberately separates **evidence estimation** from **deployment fitting**:

1. Load the exact V7.7 development component dataset and ordered predictor allowlist.
2. Use the V7.7-selected structural model family and fixed hyperparameters.
3. Recreate page-grouped out-of-fold raw structural-safety scores, so no component is scored by a model trained on its own page.
4. Fit one deployable isotonic calibrator on those pooled out-of-fold scores and development structural labels.
5. Select the frozen threshold from **calibrated out-of-fold probabilities only**, never from fitted-all-data probabilities and never from V7.8.
6. Fit one final structural model on all development components.
7. At deployment, score unseen components with the final model, then transform those raw scores through the calibrator learned from out-of-fold evidence.
8. Serialise the fitted final model, fitted calibrator, ordered schema, threshold and hashes as one policy bundle.

Out-of-fold scores remain evidence for calibration and threshold selection. They are **not** treated as a deployable model.

## Important limitation

A final model trained on all development data is not mathematically identical to the fold-specific models that generated the out-of-fold scores. The calibrator therefore maps the selected model family’s page-grouped out-of-fold scores to empirical probabilities, while the final fitted model supplies scores for unseen data. This is a conventional deployment bridge, but V7.8 remains the required independent test of transport.

The frozen policy remains a candidate for independent validation. It is not automatic-correction authorisation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy scikit-learn joblib matplotlib


In [ ]:
import json, hashlib, math, platform, sys
import numpy as np
import pandas as pd
import sklearn, joblib
from pathlib import Path
from datetime import datetime, timezone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, average_precision_score, roc_auc_score

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V77_ROOT=ROOT/'structural_risk_calibration_v7_7'
OUTPUT_ROOT=ROOT/'structural_policy_freeze_v7_7'
V77_RUN_OVERRIDE=None  # Set to the exact corrected V7.7 run used by the passing V7.8.2 audit.
RANDOM_SEED=42
REQUIRE_ZERO_OBSERVED_FALSE_SAFE=False
# If False, preserve the exact diagnostic threshold recorded in the source V7.7 manifest.
# If True, recompute the maximum-coverage zero-observed-failure threshold from recreated calibrated OOF evidence.
RECOMPUTE_THRESHOLD=False
REPLAY_ATOL=1e-12
REPLAY_RTOL=1e-12

REQUIRED=['v7_7_manifest.json','tables/asymmetric_component_dataset.csv','tables/predictor_allowlist.csv','tables/out_of_fold_asymmetric_predictions.csv','tables/candidate_structural_safety_policies.csv','tables/page_grouped_model_summary.csv']
def newest_complete(root):
 hits=[]
 if root.exists():
  for p in root.rglob('*'):
   if p.is_dir() and all((p/r).exists() for r in REQUIRED):hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
V77=Path(V77_RUN_OVERRIDE) if V77_RUN_OVERRIDE else newest_complete(V77_ROOT)
if V77 is None or not all((V77/r).exists() for r in REQUIRED):raise RuntimeError('Set V77_RUN_OVERRIDE to the exact complete corrected V7.7 run.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_policy_freeze';OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';ART=OUT/'artefacts'
for d in [OUT,TABLES,ART]:d.mkdir(parents=True,exist_ok=True)
print('Source V7.7:',V77);print('Output:',OUT)


## Hashing and fail-closed schema helpers


In [ ]:
def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def canonical_hash(value):return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,value):
 tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str));tmp.replace(path)
def wilson_upper(errors,n,z=1.96):
 if n==0:return np.nan
 p=errors/n;den=1+z*z/n;centre=p+z*z/(2*n);radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n));return float((centre+radius)/den)

def strict_predictor_frame(frame,predictors):
 missing=[c for c in predictors if c not in frame.columns]
 if missing:raise RuntimeError(f'Missing predictors: {missing}')
 X=frame.loc[:,predictors].copy()
 for c in predictors:X[c]=pd.to_numeric(X[c],errors='coerce')
 arr=X.to_numpy(dtype=float)
 if np.isinf(arr).any():raise RuntimeError('Infinite predictor value found.')
 # NaN is permitted because the frozen pipeline contains its fitted median imputer.
 return X


## Load and verify the exact V7.7 evidence chain


In [ ]:
manifest=json.loads((V77/'v7_7_manifest.json').read_text())
df=pd.read_csv(V77/'tables'/'asymmetric_component_dataset.csv')
allow=pd.read_csv(V77/'tables'/'predictor_allowlist.csv')
oof_saved=pd.read_csv(V77/'tables'/'out_of_fold_asymmetric_predictions.csv')
policy_saved=pd.read_csv(V77/'tables'/'candidate_structural_safety_policies.csv')
model_summary=pd.read_csv(V77/'tables'/'page_grouped_model_summary.csv')
predictors=allow['predictor'].astype(str).tolist()
if predictors!=list(manifest['predictors']):raise RuntimeError('Predictor allowlist and V7.7 manifest disagree in content or order.')
if manifest.get('best_structural_model')!='forest':raise RuntimeError(f"This freeze notebook is specified for the V7.7-selected forest, found {manifest.get('best_structural_model')}")
if not {'page','component','structural_safe'}.issubset(df.columns):raise RuntimeError('Development dataset lacks page/component/structural_safe.')
if df.duplicated(['page','component']).any():raise RuntimeError('Duplicate development page/component rows.')
X=strict_predictor_frame(df,predictors);y=df.structural_safe.astype(bool).astype(int);groups=df.page.astype(str)
if groups.nunique()<3 or y.nunique()!=2:raise RuntimeError('Insufficient grouped binary development data.')
source_hashes={r:sha256_file(V77/r) for r in REQUIRED}
print('Components:',len(df),'Pages:',groups.nunique(),'Predictors:',len(predictors),'Safe prevalence:',float(y.mean()))


## Recreate page-grouped out-of-fold raw scores


In [ ]:
def make_model():
 return Pipeline([
  ('impute',SimpleImputer(strategy='median')),
  ('model',RandomForestClassifier(n_estimators=800,min_samples_leaf=5,max_features='sqrt',class_weight='balanced_subsample',random_state=RANDOM_SEED,n_jobs=-1)),
 ])
splitter=LeaveOneGroupOut() if groups.nunique()<=12 else GroupKFold(n_splits=min(10,groups.nunique()))
raw_oof=np.full(len(df),np.nan);fold_rows=[]
for fold,(train,test) in enumerate(splitter.split(X,y,groups),1):
 if y.iloc[train].nunique()<2:raise RuntimeError(f'Fold {fold} training labels contain one class.')
 model=make_model();model.fit(X.iloc[train],y.iloc[train]);scores=model.predict_proba(X.iloc[test])[:,1];raw_oof[test]=scores
 fold_rows.append({'fold':fold,'test_pages':'|'.join(sorted(groups.iloc[test].unique())),'train_components':len(train),'test_components':len(test),'test_safe':int(y.iloc[test].sum()),'raw_score_min':float(scores.min()),'raw_score_max':float(scores.max())})
if not np.isfinite(raw_oof).all():raise RuntimeError('Incomplete OOF scores.')
pd.DataFrame(fold_rows).to_csv(TABLES/'page_grouped_freeze_folds.csv',index=False)
print('Raw OOF AP:',average_precision_score(y,raw_oof),'ROC AUC:',roc_auc_score(y,raw_oof),'Brier:',brier_score_loss(y,raw_oof))


## Fit deployable calibration from OOF evidence and freeze threshold


In [ ]:
# One calibrator is fitted only after every row has an OOF score. It never sees final-model in-sample scores.
calibrator=IsotonicRegression(out_of_bounds='clip',y_min=0.0,y_max=1.0,increasing=True)
calibrator.fit(raw_oof,y.to_numpy())
prob_oof=calibrator.predict(raw_oof)

threshold_from_manifest=float(manifest['diagnostic_threshold'])
if RECOMPUTE_THRESHOLD:
 candidates=[]
 for threshold in np.linspace(0,1,1001):
  selected=prob_oof>=threshold;n=int(selected.sum())
  if not n:continue
  errors=int(np.sum(selected & (y.to_numpy()==0)))
  candidates.append({'probability_threshold':float(threshold),'selected_components':n,'structural_unsafe_selected':errors,'observed_false_safe_rate':errors/n,'false_safe_rate_wilson95_upper':wilson_upper(errors,n)})
 candidate_df=pd.DataFrame(candidates).drop_duplicates(['selected_components','structural_unsafe_selected'])
 zero=candidate_df[candidate_df.structural_unsafe_selected==0]
 if zero.empty:raise RuntimeError('No zero-observed-failure threshold exists.')
 threshold=float(zero.sort_values(['selected_components','probability_threshold'],ascending=[False,False]).iloc[0].probability_threshold)
 threshold_origin='recomputed_from_recreated_page_grouped_OOF'
else:
 threshold=threshold_from_manifest;threshold_origin='exact_source_v7_7_manifest'

selected=prob_oof>=threshold;errors=int(np.sum(selected & (y.to_numpy()==0)));n=int(selected.sum())
if REQUIRE_ZERO_OBSERVED_FALSE_SAFE and errors:raise RuntimeError(f'Frozen threshold selects {errors} development structural failures.')
calibration_evidence=pd.DataFrame({'page':df.page,'component':df.component,'structural_safe':y.astype(bool),'raw_oof_score':raw_oof,'calibrated_oof_probability':prob_oof,'selected_at_frozen_threshold':selected})
calibration_evidence.to_csv(TABLES/'policy_freeze_oof_evidence.csv',index=False)
summary={'threshold':threshold,'threshold_origin':threshold_origin,'selected_components':n,'selected_fraction':n/len(df),'structural_unsafe_selected':errors,'observed_false_safe_rate':errors/n if n else None,'false_safe_rate_wilson95_upper':wilson_upper(errors,n),'structural_safe_recall':float(np.sum(selected&(y.to_numpy()==1))/max(int(y.sum()),1)),'calibrated_oof_brier':float(brier_score_loss(y,prob_oof))}
print(summary)


## Fit final all-development model and serialise the candidate policy


In [ ]:
final_model=make_model();final_model.fit(X,y)
final_raw=final_model.predict_proba(X)[:,1];final_prob=calibrator.predict(final_raw)
# These fitted-all-data scores are exported for replay diagnostics only. They are not used to choose the threshold.
replay=pd.DataFrame({'page':df.page,'component':df.component,'final_model_raw_score':final_raw,'final_model_calibrated_probability':final_prob,'decision':final_prob>=threshold})
replay.to_csv(TABLES/'final_model_development_replay.csv',index=False)

MODEL_PATH=ART/'structural_model.joblib';CALIBRATOR_PATH=ART/'isotonic_calibrator.joblib';SCHEMA_PATH=ART/'predictor_schema.json'
joblib.dump(final_model,MODEL_PATH,compress=3);joblib.dump(calibrator,CALIBRATOR_PATH,compress=3)
schema={'ordered_predictors':predictors,'allow_nan':True,'reject_infinite':True,'reject_missing':True,'ignore_non_predictor_columns':True,'target':'structural_safe'}
atomic_json(SCHEMA_PATH,schema)

policy_identity={'schema_version':'v7.7F-1','source_v7_7_hashes':source_hashes,'random_seed':RANDOM_SEED,'model_family':'RandomForestClassifier','model_parameters':{'n_estimators':800,'min_samples_leaf':5,'max_features':'sqrt','class_weight':'balanced_subsample','random_state':RANDOM_SEED},'imputer':{'strategy':'median'},'calibration':{'method':'IsotonicRegression','training_scores':'page-grouped out-of-fold raw scores','out_of_bounds':'clip'},'threshold':threshold,'threshold_origin':threshold_origin,'ordered_predictors':predictors,'development_components':len(df),'development_pages':int(groups.nunique())}
POLICY_ID=canonical_hash(policy_identity)
POLICY_PATH=ART/'policy.json'
policy={'policy_id':POLICY_ID,**policy_identity,'model_sha256':sha256_file(MODEL_PATH),'calibrator_sha256':sha256_file(CALIBRATOR_PATH),'schema_sha256':sha256_file(SCHEMA_PATH),'candidate_policy_only':True,'independent_validation_required':True,'development_oof_evidence':summary,'software':{'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,'scikit_learn':sklearn.__version__,'joblib':joblib.__version__}}
atomic_json(POLICY_PATH,policy)
print('POLICY ID:',POLICY_ID)


## Deterministic replay and bundle-integrity gate


In [ ]:
loaded_policy=json.loads(POLICY_PATH.read_text());loaded_schema=json.loads(SCHEMA_PATH.read_text());loaded_model=joblib.load(MODEL_PATH);loaded_calibrator=joblib.load(CALIBRATOR_PATH)
X_replay=strict_predictor_frame(df,loaded_schema['ordered_predictors']);raw2=loaded_model.predict_proba(X_replay)[:,1];prob2=loaded_calibrator.predict(raw2);decision2=prob2>=loaded_policy['threshold']
checks={'model_hash':loaded_policy['model_sha256']==sha256_file(MODEL_PATH),'calibrator_hash':loaded_policy['calibrator_sha256']==sha256_file(CALIBRATOR_PATH),'schema_hash':loaded_policy['schema_sha256']==sha256_file(SCHEMA_PATH),'raw_scores_replay':bool(np.allclose(final_raw,raw2,atol=REPLAY_ATOL,rtol=REPLAY_RTOL)),'probabilities_replay':bool(np.allclose(final_prob,prob2,atol=REPLAY_ATOL,rtol=REPLAY_RTOL)),'decisions_replay':bool(np.array_equal(final_prob>=threshold,decision2)),'predictor_order':loaded_schema['ordered_predictors']==predictors}
if not all(checks.values()):raise RuntimeError(f'POLICY REPLAY FAILED: {checks}')
replay_hash=canonical_hash({'raw':[float(x) for x in raw2],'probability':[float(x) for x in prob2],'decision':[bool(x) for x in decision2]})
atomic_json(ART/'replay_test.json',{'policy_id':POLICY_ID,'checks':checks,'development_replay_hash':replay_hash,'passed':True})
print('DETERMINISTIC POLICY REPLAY PASSED:',checks)


## Frozen scorer for unseen V7.8 components


In [ ]:
def apply_frozen_policy(component_frame,bundle_dir=ART):
 bundle_dir=Path(bundle_dir);policy=json.loads((bundle_dir/'policy.json').read_text());schema=json.loads((bundle_dir/'predictor_schema.json').read_text())
 model_path=bundle_dir/'structural_model.joblib';cal_path=bundle_dir/'isotonic_calibrator.joblib'
 if policy['model_sha256']!=sha256_file(model_path) or policy['calibrator_sha256']!=sha256_file(cal_path):raise RuntimeError('Policy artefact hash mismatch.')
 model=joblib.load(model_path);cal=joblib.load(cal_path);Xin=strict_predictor_frame(component_frame,schema['ordered_predictors'])
 raw=model.predict_proba(Xin)[:,1];prob=cal.predict(raw);out=component_frame.copy();out['raw_structural_safe_score']=raw;out['prob_structural_safe']=prob;out['candidate_structural_auto_safe']=prob>=float(policy['threshold']);out['policy_id']=policy['policy_id'];return out

# Schema smoke tests.
try:
 apply_frozen_policy(df.drop(columns=[predictors[0]]))
 raise RuntimeError('Missing-predictor smoke test did not fail.')
except RuntimeError as e:
 if 'Missing predictors' not in str(e):raise
print('Frozen scorer smoke tests passed.')


## Export manifest and instructions


In [ ]:
manifest_out={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.7F','source_v7_7_run':str(V77),'policy_id':POLICY_ID,'policy_bundle':str(ART),'files':{p.name:sha256_file(p) for p in sorted(ART.iterdir()) if p.is_file()},'procedure':{'model_selection':'taken from source V7.7 page-grouped OOF evidence; forest required','raw_score_evidence':'recreated with page-grouped OOF folds','calibrator_fit':'single deployable isotonic calibrator fitted to pooled page-grouped OOF raw scores and labels','threshold_selection':threshold_origin,'final_model_fit':'selected model family fitted to all development components after calibration evidence and threshold are fixed','independent_outcomes_used':False},'candidate_policy_is_production_authorisation':False,'independent_validation_required':True,'development_oof_summary':summary,'replay_checks':checks}
atomic_json(OUT/'v7_7_policy_freeze_manifest.json',manifest_out)
lines=['V7.7F DETERMINISTIC STRUCTURAL-POLICY FREEZE','',f'Policy ID: {POLICY_ID}',f'Source V7.7: {V77}',f'Predictors: {len(predictors)}',f'Development components: {len(df)} across {groups.nunique()} pages',f'Frozen threshold: {threshold}',f'Threshold origin: {threshold_origin}',f'Development OOF selected: {n}',f'Development OOF structural failures selected: {errors}',f'Development OOF false-safe Wilson 95% upper: {wilson_upper(errors,n)}','','Policy bundle:',str(ART),'','NEXT: apply the frozen bundle once to the already locked V7.8 component_dataset.csv without refitting, recalibrating or changing the threshold.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines));print(chr(10).join(lines))
